# Layer 2 (FRIENDS): validation checks

The checks of `layer2_defense/validate_friends.py`, on the CIFAR-100 / EWC victim and the poisoned task 9:

| Check | What it tests | Time (T4) |
|---|---|---|
| **V1** | our `generate_friendly_noise` gives exactly the official FRIENDS function's output (CPU: bit-for-bit) | ~5 min |
| **V4** | the training plumbing: random noise on every step and not at test time, a no-defense run equals the original `train()`, eta computed once at the right epoch and within its cap, BatchNorm modes, head-only epochs, per-epoch log | ~2 min |
| **V2** | how large the friendly noise gets, and how gentle it is to the model, for options (a) features / (b) warm-up 2 / (c) checkpoint | ~5 min |
| **V5** | 2-epoch training test: does eta protect the old tasks better than random noise of the same size? (runs V2 too) | ~5 min |
| **V6** | how much of the forgetting is BatchNorm statistics | ~3 min |
| **V7** | does the noise touch the poison's "signature" in the training signal? | ~2 min |

**Run V1 + V4 first after any code change** (and now, once: the code moved to `layer2_defense/` and `friends.py` is back to the
official FRIENDS optimizer). Both must say PASS before the option a/b/c runs. V2, V5, V6 and V7 were already run; re-run them only if you want them on this code version.

## 1. Settings

In [ ]:
EPS = '0.3'                       # '0.3' = the main table (cautious, eps 0.3); '0.1' = the weaker attack
NOISE = {
    '0.3': '/content/drive/MyDrive/brainwash/brainwash-repro/repro/sweep/noise/cifar100_ewc_cautious_eps0.3/noise_ewc_wcur_1.0_cifar100_ewc_cautious_eps0.3__delta_0.3_dataset_split_cifar100_target_task_0_attacked_task_9_noise_optim_lr_0.005__n_iters_1_n_epochs_5000_seed_0_mode_cautious__w_cur_1.0___min_acc_target_29.pkl',
    '0.1': '/content/drive/MyDrive/brainwash/brainwash-repro/repro/sweep/noise/cifar100_ewc_cautious_eps0.1/noise_ewc_wcur_1.0_cifar100_ewc_cautious_eps0.1__delta_0.1_dataset_split_cifar100_target_task_0_attacked_task_9_noise_optim_lr_0.005__n_iters_1_n_epochs_5000_seed_0_mode_cautious__w_cur_1.0___min_acc_target_39.pkl',
}
NOISE_PKL = NOISE[EPS]
# One folder per eps, SHARED by the option a / b / c notebooks: A_clean and B_poison then run only once, and each
# notebook's table shows every arm of that eps. Finished arms (a finished log in this folder) are skipped.
RESULTS_DIR = f'/content/drive/MyDrive/brainwash/layer2_runs/layer2_defense_eps{EPS}/validation'
SEED = 0                            # only seed 0: there are no victim checkpoints for other seeds yet

REPO_URL = 'https://github.com/anushkathagle/brainwash-repro.git'
REPO_BRANCH = 'layer2/Friendly'

## 2. GPU and Google Drive

In [ ]:
import torch, os
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > GPU'
print(torch.__version__, torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')
assert os.path.isfile(NOISE_PKL), f'not found: {NOISE_PKL}'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('noise pkl:', os.path.basename(NOISE_PKL), f'({os.path.getsize(NOISE_PKL)/1e6:.0f} MB)')
print('results  :', RESULTS_DIR)

## 3. Code: the `layer2/Friendly` branch

The Layer-2 code lives in the repo, in `layer2_defense/` (next to `approaches/` and `repro/`), so nothing is written here.
The first run clones the branch; re-running this cell later **pulls the newest commit** of the branch.
Private repo: add a read-only GitHub token as a Colab secret named `GH_TOKEN` (key icon in the left sidebar).

In [ ]:
import subprocess, sys
REPO_DIR = '/content/brainwash-repro'
url = REPO_URL
try:
    from google.colab import userdata
    tok = userdata.get('GH_TOKEN')
    if tok:
        url = REPO_URL.replace('https://', f'https://{tok}@')
except Exception:
    pass
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', '-b', REPO_BRANCH, url, REPO_DIR], check=True)
else:   # already cloned in this Colab session: move to the newest commit of the branch
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '-q', url, REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-q', '-B', REPO_BRANCH, 'FETCH_HEAD'], check=True)
%cd /content/brainwash-repro
assert os.path.isfile('layer2_defense/friends_layer2.py'), 'layer2_defense/ missing: is REPO_BRANCH right?'
sys.path.insert(0, '/content/brainwash-repro/layer2_defense')
import importlib, colab_helpers as H
importlib.reload(H)
H.record_code_version(RESULTS_DIR)      # which commit produced the runs in this folder -> code_version.txt

## 4. Data and checkpoints (CIFAR-100 and the authors' victim checkpoints, about 200 MB)

In [ ]:
!pip -q install huggingface_hub
!python repro/download_artifacts.py --what cifar100
!python repro/prepare_cifar100.py

## 5. The official FRIENDS code (only V1 needs it)

In [ ]:
if not os.path.isdir('/content/friendly-noise'):
    !git clone -q --depth 1 https://github.com/tianyu139/friendly-noise.git /content/friendly-noise
V = f'cd /content/brainwash-repro && python layer2_defense/validate_friends.py --noise_pkl "{NOISE_PKL}"'

## 6. V1 + V4 (must both PASS)

In [ ]:
!{V} --official_repo /content/friendly-noise --checks v1 v4 --out "{RESULTS_DIR}/v1_v4.json" 2>&1 | grep -v Warning | tee "{RESULTS_DIR}/v1_v4.log"

## 7. V2 + V5

In [ ]:
!{V} --checks v5 --out "{RESULTS_DIR}/v2_v5.json" 2>&1 | grep -v Warning | tee "{RESULTS_DIR}/v2_v5.log"

## 8. V6 (BatchNorm statistics)

In [ ]:
!{V} --checks v6 --out "{RESULTS_DIR}/v6.json" 2>&1 | grep -v Warning | tee "{RESULTS_DIR}/v6.log"

## 9. V7 (the poison's signature)

In [ ]:
!{V} --checks v7 --out "{RESULTS_DIR}/v7.json" 2>&1 | grep -v Warning | tee "{RESULTS_DIR}/v7.log"